# Smoke test

Query the new Aura DB after bulk_reingest finishes.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.generation import GraphRAG

driver = GraphDatabase.driver(
    os.environ['NEO4J_NEW_URI'],
    auth=(os.environ['NEO4J_NEW_USERNAME'], os.environ['NEO4J_NEW_PASSWORD'])
)
DB = os.environ.get('NEO4J_NEW_DATABASE', 'neo4j')
embedder = OpenAIEmbeddings(model='text-embedding-3-small')

## Pattern 1: Pure vector retrieval

In [ ]:
vr = VectorRetriever(driver=driver, index_name='chunk_embedding', embedder=embedder, neo4j_database=DB)
for item in vr.search(query_text='wrong-way risk in XVA close-out', top_k=5).items:
    print(item.content[:200], '...')
    print('  score:', item.metadata.get('score'))
    print()

## Pattern 2: Vector + Cypher filter (scoped to topic)

In [ ]:
rq = '''
MATCH (chunk)-[:FROM_DOCUMENT]->(doc:Document)<-[:HAS_DOCUMENT]-(p:Paper)
MATCH (p)-[:HAS_TOPIC]->(t:Topic)
WHERE t.name = $topic
RETURN chunk.text AS text, p.id AS paper_id, p.title AS title
'''
vcr = VectorCypherRetriever(driver=driver, index_name='chunk_embedding', retrieval_query=rq, embedder=embedder, neo4j_database=DB)
for item in vcr.search(query_text='exponential change of measure', query_params={'topic': 'deep-bsde-methods-for-xva'}, top_k=5).items:
    print(item.content[:200], '...')
    print()

## Pattern 3: Pure Cypher graph traversal

In [ ]:
with driver.session(database=DB) as s:
    for r in s.run('''
        MATCH (c:Concept)-[:DERIVED_FROM]->(p:Paper)
        RETURN c.name AS concept, count(DISTINCT p) AS papers
        ORDER BY papers DESC LIMIT 10
    '''):
        print(r['papers'], r['concept'])

## Pattern 4: Full GraphRAG (retrieval + LLM synthesis)

In [ ]:
llm = OpenAILLM(model_name='gpt-4o-mini')
rag = GraphRAG(retriever=vr, llm=llm)
resp = rag.search(
    query_text='How does the close-out convention affect XVA pricing under counterparty default?',
    return_context=True,
)
print(resp.answer)

## Pattern 5: Actual GraphRAG

Vector RAG with structural context: retrieved chunks are **enriched** with their paper's authors, topics, derived concepts, citations — and that structure goes into the LLM prompt, not just the chunk text.

This implements [Han et al. 2024]'s 5-component GraphRAG framework: query → retriever (vector + Cypher) → organizer (assemble structured context) → generator (LLM with structural prompt). The retrieval style is closest to **LightRAG** (Guo et al. 2024) — flat graph, multi-faceted entity/relation context — adapted to use only your existing schema (no entity/relation vector indexes yet).


In [ ]:
# Build the structured context inside Cypher as a single string per chunk.
# That way the standard GraphRAG class can use it without custom plumbing —
# the LLM sees: paper title, authors, topics, derived concepts, citations,
# related ideas, then the chunk text.
GRAPH_RETRIEVAL_QUERY = '''
WITH node AS chunk, score
MATCH (chunk)-[:FROM_DOCUMENT]->(doc:Document)<-[:HAS_DOCUMENT]-(p:Paper)
OPTIONAL MATCH (a:Author)-[:AUTHORED]->(p)
WITH chunk, score, p, collect(DISTINCT a.name)[0..6] AS authors
OPTIONAL MATCH (p)-[:HAS_TOPIC]->(t:Topic)
WITH chunk, score, p, authors, collect(DISTINCT t.name)[0..5] AS topics
OPTIONAL MATCH (c:Concept)-[:DERIVED_FROM]->(p)
WITH chunk, score, p, authors, topics, collect(DISTINCT c.name)[0..8] AS concepts
OPTIONAL MATCH (p)-[:CITES]->(cited:Paper)
WITH chunk, score, p, authors, topics, concepts, collect(DISTINCT cited.title)[0..3] AS cited_titles
OPTIONAL MATCH (p)<-[:EVIDENCED_BY]-(idea:Idea)
WITH chunk, score, p, authors, topics, concepts, cited_titles, collect(DISTINCT idea.title)[0..3] AS related_ideas
RETURN
  '[Source paper] '       + coalesce(p.title, '?')                          + '\n' +
  '[Authors] '            + apoc.text.join(authors, ', ')                   + '\n' +
  '[Topics] '             + apoc.text.join(topics, ', ')                    + '\n' +
  '[Concepts derived] '   + apoc.text.join(concepts, ', ')                  + '\n' +
  '[Cites] '              + apoc.text.join(cited_titles, '; ')              + '\n' +
  '[Related ideas] '      + apoc.text.join(related_ideas, '; ')             + '\n' +
  '[Passage]\n'          + coalesce(chunk.text, '')                        AS info,
  score
ORDER BY score DESC
'''

vcr_graph = VectorCypherRetriever(
    driver=driver,
    index_name='chunk_embedding',
    retrieval_query=GRAPH_RETRIEVAL_QUERY,
    embedder=embedder,
    neo4j_database=DB,
)


In [ ]:
# Standard GraphRAG, but the retriever is graph-aware: each retrieved 'item'
# already contains the paper's structural metadata in its text body.
rag_graph = GraphRAG(retriever=vcr_graph, llm=llm)
resp = rag_graph.search(
    query_text='How does the close-out convention affect XVA pricing under counterparty default?',
    return_context=True,
)
print(resp.answer)
print()
print('--- retrieved sources ---')
for it in resp.retriever_result.items[:3]:
    print(str(it.content)[:400], '...\n')


## Pattern 6: real graph traversal

Query → entities → graph seeds → multi-hop traversal → reachable papers → vector-rank chunks within those papers → LLM synthesises with the traversal evidence as context.

This is **proper subgraph expansion** ala [LightRAG] / [HippoRAG] — chunks are picked because their papers are *structurally* close to query entities in the curated graph, not just because their text resembles the query string. Pattern 5 walked exactly one edge (chunk→paper); this walks up to three (`Concept` → `RELATED_TO`/`BELONGS_TO`/etc. → `Concept`/`Topic` → ... → `Paper`), so it can reach papers that share concepts with your query without ever mentioning your query terms verbatim.


In [ ]:
import json, re
from openai import OpenAI
oai = OpenAI()

def extract_keywords(q: str) -> list[str]:
    """LLM-extracted atomic terms + capitalised-word tokens from the raw query."""
    r = oai.chat.completions.create(
        model='gpt-4o-mini', response_format={'type': 'json_object'},
        messages=[{'role': 'user', 'content':
            f'Extract atomic technical terms from this question. Each must be 1-2 words MAX. '
            f'GOOD: "XVA", "close-out", "Schrödinger bridge", "rectified flow". '
            f'BAD (too long): "close-out convention", "XVA pricing under default". '
            f'Return JSON {{"keywords": [...]}}.\nQuestion: {q}'}])
    kws = json.loads(r.choices[0].message.content).get('keywords', [])
    stopwords = {'how','does','the','affect','under','what','which','are','is','of','to','and','or','a','an','for','in','on','from','at','with','by','it','that','these','those','this'}
    tokens = [t for t in re.findall(r'[A-Za-z][A-Za-z\-]{2,}', q) if t.lower() not in stopwords]
    return list(set(kws + tokens))

def find_seeds(session, q: str, kws: list[str]) -> list[dict]:
    """Locate Concept/Topic nodes either matching extracted keywords OR appearing in the query verbatim."""
    a = list(session.run('''
        UNWIND $keywords AS kw
        MATCH (n) WHERE (n:Concept OR n:Topic) AND toLower(n.name) CONTAINS toLower(kw)
        RETURN DISTINCT n.name AS name, head(labels(n)) AS label
    ''', keywords=kws))
    b = list(session.run('''
        MATCH (m) WHERE (m:Concept OR m:Topic) AND size(m.name) > 3 AND toLower($q_text) CONTAINS toLower(m.name)
        RETURN DISTINCT m.name AS name, head(labels(m)) AS label
    ''', q_text=q))
    return [{'name': n, 'label': l} for l, n in {(r['label'], r['name']) for r in (a + b)}]

def reachable_papers(session, seed_names: list[str]) -> list[dict]:
    """Walk up to 3 hops from each seed, excluding chunk/document edges. Return papers ranked by seed coverage."""
    return list(session.run('''
        UNWIND $seed_names AS sn
        MATCH (seed) WHERE (seed:Concept OR seed:Topic) AND seed.name = sn
        MATCH path = (seed)-[r*1..3]-(p:Paper)
        WHERE NONE(rel IN r WHERE type(rel) IN ['FROM_CHUNK','FROM_DOCUMENT','HAS_DOCUMENT','NEXT_CHUNK','HAS_SUMMARY'])
        WITH p, seed, length(path) AS hops
        WITH p, min(hops) AS min_hops, count(DISTINCT seed) AS seed_count, collect(DISTINCT seed.name)[0..3] AS via
        RETURN p.id AS paper_id, p.title AS title, min_hops, seed_count, via
        ORDER BY seed_count DESC, min_hops ASC
        LIMIT 25
    ''', seed_names=seed_names))

def hybrid_chunks(session, paper_ids: list[str], query_emb: list[float], top_k: int = 5) -> list[dict]:
    """Vector-search chunks from candidate pool (top-50 by sim), then keep only those in graph-reachable papers."""
    return list(session.run('''
        CALL db.index.vector.queryNodes('chunk_embedding', 50, $qv) YIELD node AS chunk, score
        MATCH (chunk)-[:FROM_DOCUMENT]->(:Document)<-[:HAS_DOCUMENT]-(p:Paper)
        WHERE p.id IN $paper_ids
        RETURN chunk.text AS text, p.title AS title, p.id AS paper_id, score
        ORDER BY score DESC
        LIMIT $top_k
    ''', qv=query_emb, paper_ids=paper_ids, top_k=top_k))


In [ ]:
def graph_traversal_rag(query: str, top_k: int = 5, verbose: bool = True) -> str:
    keywords = extract_keywords(query)
    if verbose: print(f'[1] keywords: {keywords}')

    with driver.session(database=DB) as s:
        seeds = find_seeds(s, query, keywords)
    if verbose:
        print(f'[2] seeds in graph: {len(seeds)}')
        for sd in seeds[:6]: print(f'    ({sd["label"]}) {sd["name"]}')

    if not seeds:
        if verbose: print('No graph seeds — falling back to plain vector RAG.')
        # graceful degradation
        emb = oai.embeddings.create(model='text-embedding-3-small', input=query).data[0].embedding
        with driver.session(database=DB) as s:
            chunks = list(s.run('''
                CALL db.index.vector.queryNodes('chunk_embedding', $top_k, $qv) YIELD node, score
                MATCH (node)-[:FROM_DOCUMENT]->(:Document)<-[:HAS_DOCUMENT]-(p:Paper)
                RETURN node.text AS text, p.title AS title, score ORDER BY score DESC
            ''', qv=emb, top_k=top_k))
        papers = []
    else:
        with driver.session(database=DB) as s:
            papers = reachable_papers(s, [sd['name'] for sd in seeds])
        if verbose:
            print(f'\n[3] papers reachable in <=3 hops: {len(papers)}')
            for p in papers[:5]: print(f'    hops={p["min_hops"]} via {p["seed_count"]} seeds  {(p["title"] or "?")[:60]}')

        emb = oai.embeddings.create(model='text-embedding-3-small', input=query).data[0].embedding
        with driver.session(database=DB) as s:
            chunks = hybrid_chunks(s, [p['paper_id'] for p in papers], emb, top_k=top_k)
        if verbose:
            print(f'\n[4] top-{top_k} chunks (graph × vector):')
            for c in chunks: print(f'    sim={c["score"]:.3f}  {(c["title"] or "?")[:55]}')

    # === GENERATOR: structural prompt with both chunk text AND traversal evidence ===
    paths_evidence = '\n'.join([f'- {p["title"]} (reached via {p["seed_count"]} seed concept(s): {", ".join(p["via"])}; {p["min_hops"]} hop(s))' for p in papers[:8]])
    chunk_blocks   = '\n\n'.join([f'[Source: {c["title"]}]\n{c["text"]}' for c in chunks])
    prompt = f'''You are answering a research question using evidence from a curated knowledge graph.

Query keywords located in graph: {", ".join(keywords)}
Seed concepts/topics: {", ".join([sd["name"] for sd in seeds[:8]])}

Papers reachable from those seeds (graph-traversal evidence — these are STRUCTURALLY connected to the query, not just textually similar):
{paths_evidence or "(no graph reachability — fell back to vector retrieval)"}

Top chunks:
{chunk_blocks}

Question: {query}

Answer in detail, citing specific papers by title. If the traversal evidence and chunk text point to different aspects of the answer, synthesise both.'''
    return oai.chat.completions.create(model='gpt-4o-mini', messages=[{'role': 'user', 'content': prompt}]).choices[0].message.content

answer = graph_traversal_rag(
    'How does the close-out convention affect XVA pricing under counterparty default?',
    top_k=5,
)
print('\n=== ANSWER ===')
print(answer)
